## Dashboards!
Create a dashboard in Saturn, based on code you've written in a Jupyter notebook.

### From notebook
If you run all the cells in this notebook, the final cell will display the dashboard inline.

### Deployment
To deploy, use this for the command go back to your project page (where you started the jupyter server). From there you can click "Create a Deployment" and use the following command:

```bash
panel serve dashboard.ipynb --port=8000 --address="0.0.0.0" --allow-websocket-origin="*"
```

In [ ]:
import numpy as np
import dask.dataframe as dd

import hvplot.dask  # noqa
import hvplot.pandas  # noqa
import panel as pn

import s3fs

TAXI_PATH = "s3://saturn-public-data/nyc-taxi"
DATA_PATH = f"{TAXI_PATH}/data/dashboard"
fs = s3fs.S3FileSystem(anon=True)

## Construct the visuals

In [ ]:
pickup_by_zone = dd.read_csv(f"{DATA_PATH}/pickup_grouped_by_zone.csv", storage_options={"anon": True})
total_rides = pickup_by_zone.total_rides.sum().compute()
total_fare = pickup_by_zone.total_fare.sum().compute()

def kpi_box(title, color, value, unit=""):
    if value > 1e9:
        value /= 1e9
        increment = "B"
    elif value > 1e6:
        value /= 1e6
        increment = "M"
    elif value > 1e3:
        value /= 1e3
        increment = "K"
    else:
        increment = ""

    return pn.pane.Markdown(
        f"""
        ### {title}
        # {unit}{value :.02f} {increment}
        """,
        style={
            "background-color": "#F6F6F6",
            "border": "2px solid black",
            "border-radius": "5px",
            "padding": "10px",
            "color": color,
        },
    )

fares = kpi_box("Total Fares", "#10874a", total_fare, "$")
rides = kpi_box("Total Rides", "#7a41ba", total_rides)
average = kpi_box("Average Fare", "coral", (total_fare / total_rides), "$")

In [ ]:
pickup_by_time = dd.read_csv(f"{DATA_PATH}/pickup_grouped_by_time.csv", storage_options={"anon": True})

def heatmap(C, data, **kwargs):
    return data.hvplot.heatmap(
        y="pickup_weekday",
        x="pickup_hour",
        C=C,
        hover_cols=["total_rides"] if C == "average_fare" else ["average_fare"],
        yticks=[
            (0, "Mon"),
            (1, "Tues"),
            (2, "Wed"),
            (3, "Thur"),
            (4, "Fri"),
            (5, "Sat"),
            (6, "Sun"),
        ],
        responsive=True,
        min_height=200,
        colorbar=False,
        **kwargs,
    ).opts(toolbar=None, padding=0)

tip_heatmap = heatmap(
    data=pickup_by_time,
    C="average_percent_tip",
    cmap="coolwarm",
    clim=(12, 18),
    title="Average Tip %",
)

In [ ]:
tip_timeseries = dd.read_csv(
    f"{DATA_PATH}/pickup_average_percent_tip_timeseries.csv",
    storage_options={"anon": True}
)
tip_timeseries = tip_timeseries.set_index(tip_timeseries.pickup_datetime.astype(np.datetime64)).compute()

date_range_slider = pn.widgets.DateRangeSlider(
    name="Show between",
    start=tip_timeseries.index[0],
    end=tip_timeseries.index[-1],
    value=(tip_timeseries.index.min(), tip_timeseries.index.max()),
)

discrete_slider = pn.widgets.DiscreteSlider(
    name="Rolling window",
    options=["1H", "2H", "4H", "6H", "12H", "1D", "2D", "7D", "14D", "1M"],
    value="1D",
)

def tip_plot(xlim, window):
    data = tip_timeseries.rolling(window).mean()
    return data.hvplot(
        y="percent_tip", xlim=xlim, ylim=(10, 18), responsive=True, min_height=200
    ).opts(toolbar="above")

tip_timeseries_plot = pn.pane.HoloViews(tip_plot(date_range_slider.value, discrete_slider.value))

def trim(target, event):
    target.object = tip_plot(event.new, discrete_slider.value)

def roll(target, event):
    target.object = tip_plot(date_range_slider.value, event.new)

discrete_slider.link(tip_timeseries_plot, callbacks={"value": roll})
date_range_slider.link(tip_timeseries_plot, callbacks={"value": trim})

In [ ]:
with open("pickup_map.html") as f:
    pickup_map = pn.pane.HTML(f.read(), min_height=500, min_width=500)

In [ ]:
dashboard_intro = """
# NYC Taxi Data

This dashboard demonstrates one mechanism for displaying summary statistics of 
the [NYC Taxi Dataset](https://www1.nyc.gov/site/tlc/about/tlc-trip-record-data.page).
This particular page uses data from 2017 to 2020.

## Why Use Dashboards?

Dashboards provide a simple alternative to notebooks that can be more easily digested
by less technical audiences. A mixture of visualizations, text, and tables lets the reader
explore the data in a guided manner without having to write code.
"""

about_saturn = """
## Deploying in Saturn Cloud

This example uses [Panel](https://panel.holoviz.org) to create a deployable interactive dashboard. 
In Saturn it is equally easy to create a dashboard using any of the other popular dashboarding
libraries such as: voila, dash, and bokeh. Learn more about how to deploy models and dashboards
in our [documentation](https://www.saturncloud.io/docs/concepts/projects/deployments).
"""
logo_file = "/tmp/logo.svg"
fs.get(f"{DATA_PATH}/saturn_logo.svg", logo_file)
logo = pn.pane.SVG(logo_file, style={"float": "right"})


dashboard = pn.GridSpec(
    name="dashboard", sizing_mode="stretch_both", min_width=800, min_height=600, max_height=850
)
dashboard[0:5, :3] = pn.Column(
    dashboard_intro,
    tip_heatmap,
    about_saturn
) 
dashboard[0, 3] = fares
dashboard[0, 4] = rides
dashboard[0, 5] = average
dashboard[0, 6] = logo
dashboard[1:5, 3:6] = pickup_map
dashboard[5:8, 0:2] = pn.Column(
    date_range_slider,
    discrete_slider,
    "*Use widgets to control rolling window average on the timeseries plot or and to restrict to between certain dates*",
)
dashboard[5:7, 2:6] = tip_timeseries_plot

dashboard.servable(title="Saturn Taxi")